In [1]:
import pandas as pd
import numpy as np

In [2]:
# ── Load data ──────────────────────────────────────────────────────────────────

df = pd.read_excel("Input_TTF_NG_Real_Average_Prices.xlsx")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

horizons = [1, 3, 6, 9, 12, 15, 18, 21, 24]

In [3]:
# ── RW with drift forecasts ────────────────────────────────────────────────────

# Drift = recursively estimated average monthly log change up to forecast origin t
# i.e. mean of [log(P_t) - log(P_{t-1})] for all available observations up to t

# h-step forecast: R_hat(t+h) = R_t * exp(drift_t * h)
# (standard log-price RW with drift, compounded over h months)

records = []

for i in range(1, len(df)):   # start at 1 — need at least one lag to estimate drift
    origin_date  = df.loc[i, "date"]
    origin_price = df.loc[i, "price_real"]

    # Recursive drift: average log return using all data up to and including t
    log_returns  = np.diff(np.log(df.loc[:i, "price_real"].values))
    drift        = log_returns.mean()

    for h in horizons:
        # h-step forecast
        forecast = origin_price * np.exp(drift * h)

        # Actual: price h months ahead
        target_idx = i + h
        actual = df.loc[target_idx, "price_real"] if target_idx < len(df) else np.nan

        # Target month label
        actual_month = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")

        records.append({
            "forecast_origin": origin_date,
            "horizon":         h,
            "model":           "RW_drift",
            "actual_month":    actual_month,
            "forecast":        forecast,
            "actual":          actual
        })

out = pd.DataFrame(records).dropna(subset=["actual"])

In [4]:
# ── Save ──────────────────────────────────────────────────────────────────────

out.to_excel("rw_drift_forecasts_long.xlsx", index=False)
print(f"Saved {len(out)} rows")
print(out.head(18).to_string(index=False))

Saved 2033 rows
forecast_origin  horizon    model actual_month    forecast    actual
     2006-03-31        1 RW_drift      2006-04   35.678004 24.771643
     2006-03-31        3 RW_drift      2006-06   48.741787 22.503165
     2006-03-31        6 RW_drift      2006-09   77.831031 18.320474
     2006-03-31        9 RW_drift      2006-12  124.280823 18.996647
     2006-03-31       12 RW_drift      2007-03  198.451989 11.842205
     2006-03-31       15 RW_drift      2007-06  316.888730 11.939590
     2006-03-31       18 RW_drift      2007-09  506.008871 19.253499
     2006-03-31       21 RW_drift      2007-12  807.996478 25.661291
     2006-03-31       24 RW_drift      2008-03 1290.211191 26.245298
     2006-04-30        1 RW_drift      2006-05   24.125837 22.748601
     2006-04-30        3 RW_drift      2006-07   22.884296 22.787489
     2006-04-30        6 RW_drift      2006-10   21.140745 16.309698
     2006-04-30        9 RW_drift      2007-01   19.530035 15.524584
     2006-04-30   